# GAN for SQLi — Chạy lại 13 run Phase 2B bị gián đoạn (78–82/200 epoch) — SeqGAN Improved — batch-size x6 — GPU

Notebook này chạy **hoàn toàn độc lập** (cùng văn phong `GAN_SQLi_Colab_SeqGAN_Improved_refinement_allR_GPU.ipynb`), dựa vào project đã có sẵn trong Drive (đã chạy `prepare-data` + `prepare-phase2b` từ trước — notebook này **không** gọi lại hai bước đó).

## Danh sách 13 run mục tiêu

13 run sau bị dừng giữa chừng ở epoch 78–82/200 (nhiều khả năng do Colab ngắt session), tất cả đều thuộc variant `max_len=160` (V2, V4, V8):

| Run | Epoch dừng |
|---|---:|
| R100 boolean/E/V2 | 78 |
| R100 time/D/V8 | 79 |
| R100 union/D/V2 | 78 |
| R200 union/D/V2 | 80 |
| R500 boolean/D/V8 | 81 |
| R500 boolean/E/V2 | 82 |
| R500 error/B/V8 | 81 |
| R500 error/D/V4 | 82 |
| R500 error/D/V8 | 82 |
| R500 time/A/V8 | 81 |
| R500 time/D/V8 | 81 |
| R500 union/D/V2 | 81 |
| R500 union/D/V8 | (dòng cuối, epoch dừng không rõ trong bảng gốc — vẫn được train lại đầy đủ) |

## Thay đổi hyperparameter — CHỈ `batch-size` x6 (64 → 384)

- `batch-size`: **64 → 384** (x6). Batch-size scale tuyến tính với bộ nhớ GPU nên an toàn để nhân đúng 6 lần trên GPU L4 (24GB VRAM).
- `hidden-dim`, `embed-dim`, `disc-embed-dim`: **giữ nguyên** (32/32/64) — các tham số này ảnh hưởng bộ nhớ theo bậc hai (O(n²) do ma trận trọng số), nên không tăng để tránh nổ bộ nhớ và để không đổi capacity của model.
- `rollout-num`: **giữ nguyên** (16) — rollout trong `rollout.py` chạy tuần tự (Python for-loop, không nhân batch), nên tăng rollout-num không giúp dùng nhiều GPU RAM hơn, chỉ làm mỗi epoch chậm hơn tuyến tính. Với `max_len=160` (toàn bộ 13 run), phần rollout đã là bottleneck thời gian lớn nhất; tăng thêm sẽ phản tác dụng với mục tiêu tốc độ.

**⚠️ 13 run này KHÔNG còn hyperparameter giống hệt các run khác trong ma trận Phase 2B chính thức** (batch-size khác). Kết quả được ghi vào `variant-id` gắn hậu tố `_B6X` (ví dụ `V2_B6X`) và một thư mục `phase2b_batch6x` **riêng**, tách khỏi `results/phase2b/` chuẩn, để không lẫn vào `select-ratio`/`finalize` một cách vô tình. Nếu muốn dùng chung ranking chính thức, cần tự cân nhắc lại.

## Không có checkpoint từ lần chạy trước

13 run bị dừng ở epoch 78-82 không có file `adversarial_epoch_*.pt` được lưu lại, nên **phải train lại từ epoch 0/200**, không resume được vào giữa. Notebook này bật `--checkpoint-dir` cho lần chạy này, để nếu Colab ngắt session lần nữa, bạn có thể resume thật bằng `--resume-latest` (xem mục 7).


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Copy project — trỏ đúng vào project `GAN_for_SQLi` đã có sẵn `data/prepared/phase2b/` (đã chạy `prepare-data` + `prepare-phase2b` từ trước), không đụng `results/phase2b/` gốc

In [ ]:
import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import psutil
import torch
import yaml

# Doi duong dan nay cho dung project GAN_for_SQLi chinh cua ban tren Drive
# (project da co san data/prepared/phase2b/dataset_manifest.json tu lan prepare-phase2b truoc).
PATH_COLAB = "/content/drive/MyDrive/GAN/GAN_for_SQLi/"

DRIVE_PROJECT = Path(PATH_COLAB.rstrip("/"))
RESULTS_ROOT = DRIVE_PROJECT / "result"
LOCAL_PROJECT = Path("/content/GAN_for_SQLi_rerun_13")


assert DRIVE_PROJECT.is_dir(), f"Khong tim thay folder Drive: {DRIVE_PROJECT}"
assert (DRIVE_PROJECT / "scripts/research_pipeline.py").is_file(), (
    f"Khong tim thay project tai {DRIVE_PROJECT}"
)

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    DRIVE_PROJECT,
    LOCAL_PROJECT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(
        "__pycache__", "*.pyc", "results", "results_smoke", "results_mini",
    ),
)
os.chdir(LOCAL_PROJECT)

print("Project (local):", LOCAL_PROJECT)
print("Results (ghi vao Drive):", RESULTS_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Can runtime GPU (mong doi L4, 24GB VRAM)"

PHASE2B_MANIFEST = LOCAL_PROJECT / "data" / "prepared" / "phase2b" / "dataset_manifest.json"
if PHASE2B_MANIFEST.exists():
    print("Da tim thay Phase 2B dataset manifest:", PHASE2B_MANIFEST)
else:
    print("Chua co Phase 2B dataset manifest -- se tu dong tao lai o buoc sau (khong can prepare-phase2b thu cong).")


Project (local): /content/GAN_for_SQLi_rerun_13
Results (ghi vao Drive): /content/drive/MyDrive/GAN/GAN_for_SQLi/result
GPU: NVIDIA L4
Da tim thay Phase 2B dataset manifest: /content/GAN_for_SQLi_rerun_13/data/prepared/phase2b/dataset_manifest.json


## 3. Cài dependency

In [ ]:
%pip install -q -r requirements.txt


## 4. 13 run cố định cần chạy lại — đúng bảng gốc bị dừng giữa epoch 78–82/200

In [ ]:
# (ratio, family, scenario, variant) -- 13 dong, khop 1-1 voi bang cac run bi dung giua chung
TARGET_RUNS = [
    ("100", "boolean", "E", "V2"),
    ("100", "time",    "D", "V8"),
    ("100", "union",   "D", "V2"),
    ("200", "union",   "D", "V2"),
    ("500", "boolean", "D", "V8"),
    ("500", "boolean", "E", "V2"),
    ("500", "error",   "B", "V8"),
    ("500", "error",   "D", "V4"),
    ("500", "error",   "D", "V8"),
    ("500", "time",    "A", "V8"),
    ("500", "time",    "D", "V8"),
    ("500", "union",   "D", "V2"),
    ("500", "union",   "D", "V8"),
]
assert len(TARGET_RUNS) == 13
TARGET_QUADS = set(TARGET_RUNS)
print(f"{len(TARGET_RUNS)} run co dinh can chay lai")
pd.DataFrame(TARGET_RUNS, columns=["ratio", "family", "scenario", "variant_id"])


13 run co dinh can chay lai


,ratio,family,scenario,variant_id
0,100,boolean,E,V2
1,100,time,D,V8
2,100,union,D,V2
3,200,union,D,V2
4,500,boolean,D,V8
5,500,boolean,E,V2
6,500,error,B,V8
7,500,error,D,V4
8,500,error,D,V8
9,500,time,A,V8


## 5. Cấu hình runtime — chỉ đổi `batch-size` seqgan_improved thành `384` (x6 so với gốc `64`), mọi tham số khác giữ nguyên như `configs/experiment_config.yaml`

`research_pipeline.py` build lệnh train từ `configs/experiment_config.yaml` nhưng **không có khóa `batch_size` riêng cho `seqgan_improved`** trong file config gốc (batch-size của SeqGAN nằm cố định trong `models/seqgan_improved/preprocessing/config.py`, không đọc từ YAML) — vì vậy không thể chỉnh qua runtime config như các block `gan`/`ctgan`. Thay vào đó, cell dưới build lệnh train trực tiếp cho từng run bằng module `models.seqgan_improved.runtime.train`, thêm cờ `--batch-size 384`, và tự quản out-dir/checkpoint riêng — không dùng `run-matrix` cho 13 run này.

In [ ]:
base_config = yaml.safe_load(
    (LOCAL_PROJECT / "configs/experiment_config.yaml").read_text(encoding="utf-8")
)
config = json.loads(json.dumps(base_config))

# results_root van tro vao Drive, dung cho viec doc dataset/holdout path co san
config["outputs"]["results_root"] = str(RESULTS_ROOT)

RUNTIME_CONFIG_DIR = Path("/content/colab_configs")
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_CONFIG_DIR / "experiment_rerun_13.yaml"
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

PIPELINE = [sys.executable, "scripts/research_pipeline.py", "--config", str(RUNTIME_CONFIG)]

# Hyperparameter rieng cho 13 run nay -- CHI batch-size doi, con lai giu nguyen gia tri goc
BASE_BATCH_SIZE = 64
NEW_BATCH_SIZE = BASE_BATCH_SIZE * 6  # 384 -- tuyen tinh voi bo nho, an toan tren L4 24GB
VARIANT_SUFFIX = "_B6X"

BLOCK = config["generation"]["seqgan_improved"]
VARIANT_LOOKUP = {v["id"]: v for v in config["phase3"]["variants"]}

print("Runtime config:", RUNTIME_CONFIG)
print("results_root ->", config["outputs"]["results_root"])
print(f"batch-size: {BASE_BATCH_SIZE} -> {NEW_BATCH_SIZE} (hidden-dim/embed-dim/rollout-num giu nguyen)")


Runtime config: /content/colab_configs/experiment_rerun_13.yaml
results_root -> /content/drive/MyDrive/GAN/GAN_for_SQLi/result
batch-size: 64 -> 384 (hidden-dim/embed-dim/rollout-num giu nguyen)


## 5b. Tu dong tao lai `dataset_manifest.json` cho Phase 2B neu bi mat (khong can chay lai prepare-data/rank-phase2a/prepare-phase2b thu cong)

`prepare-phase2b` la buoc xu ly **deterministic** (cung `seed` co dinh) tren cac file `data/prepared/splits/{family}_train_pool.csv` da co san trong repo. Neu manifest bi mat (vi du do mat session truoc), cell duoi tu dong tinh lai **chi cho dung 7 cap (family, scenario)** ma 13 run da chot (`TARGET_RUNS`) can toi -- khong can file `top2_scenarios_per_family.csv` cua buoc rank-phase2a, va khong dung toi rang buoc "du 2 scenario/family" cua pipeline goc. Ket qua duoc luu ca vao local va vao Drive de khong bi mat lan nua.

In [ ]:
import math
import shutil
from common.ingestion import read_records, write_records, scenario_order, write_json

if PHASE2B_MANIFEST.exists():
    print("Manifest phase2b da co san, bo qua buoc tao lai:", PHASE2B_MANIFEST)
else:
    print("Khong tim thay manifest phase2b -- tu dong tao lai tu 13 run da chot (TARGET_RUNS).")

    # Chi lay dung nhung cap (family, scenario, ratio) ma 13 run thuc su can -- khong dong toi cai gi khac
    needed_triples = sorted({(family, scenario, ratio) for ratio, family, scenario, _ in TARGET_RUNS})
    print(f"Can tao lai {len(needed_triples)} cap (family, scenario, ratio) duy nhat tu {len(TARGET_RUNS)} run:")
    for t in needed_triples:
        print("  ", t)

    SPLITS_DIR = LOCAL_PROJECT / "data" / "prepared" / "splits"
    PHASE2B_DIR = LOCAL_PROJECT / "data" / "prepared" / "phase2b"
    PHASE2B_DIR.mkdir(parents=True, exist_ok=True)

    seed = int(base_config["seed"])
    normal_train_count = len(read_records(SPLITS_DIR / "normal_train.csv"))

    pool_cache = {}
    order_cache = {}
    rows = []
    for family, scenario, ratio in needed_triples:
        full = ratio == "full"
        if family not in pool_cache:
            pool_cache[family] = read_records(SPLITS_DIR / f"{family}_train_pool.csv")
        pool = pool_cache[family]

        cache_key = (family, scenario, full)
        if cache_key not in order_cache:
            order_cache[cache_key] = scenario_order(pool, scenario, seed, full=full)
        candidates, capacity = order_cache[cache_key]

        target = len(pool) if full else math.floor(normal_train_count / int(ratio))
        feasible = capacity >= target
        output = PHASE2B_DIR / family / scenario / f"R{ratio}" / "attack_train.csv"
        if feasible:
            selected = sorted(candidates[:target], key=lambda record: record.order)
            write_records(output, selected)
        rows.append({
            "family": family, "scenario": scenario, "ratio": str(ratio),
            "target": target, "capacity": capacity,
            "selected": target if feasible else 0,
            "status": "ready" if feasible else "insufficient_pool",
            "output": str(output.relative_to(LOCAL_PROJECT)).replace("\\", "/"),
        })

    not_ready = [r for r in rows if r["status"] != "ready"]
    assert not not_ready, f"Cac cap sau khong du du lieu trong pool (insufficient_pool): {not_ready}"

    manifest = {
        "seed": seed,
        "selection": "regenerated_directly_from_TARGET_RUNS_no_phase2a_selection_file",
        "cells": len(needed_triples),
        "ratios": sorted({r for _, _, r in needed_triples}),
        "rows": rows,
    }
    write_json(PHASE2B_MANIFEST, manifest)
    ready_count = sum(1 for r in rows if r["status"] == "ready")
    print(f"Da tao lai manifest phase2b: {PHASE2B_MANIFEST}  ({ready_count}/{len(rows)} san sang)")

    # Luu ban sao ve Drive de KHONG bi mat lan nua
    drive_phase2b_dir = DRIVE_PROJECT / "data" / "prepared" / "phase2b"
    shutil.copytree(PHASE2B_DIR, drive_phase2b_dir, dirs_exist_ok=True)
    print("Da luu ban sao phase2b (data + manifest) vao Drive:", drive_phase2b_dir)


Manifest phase2b da co san, bo qua buoc tao lai: /content/GAN_for_SQLi_rerun_13/data/prepared/phase2b/dataset_manifest.json


## 6. Đọc đường dẫn dataset của 13 cell từ `data/prepared/phase2b/dataset_manifest.json` đã có sẵn (không chạy lại `prepare-phase2b`)

In [ ]:
phase2b_manifest = json.loads(PHASE2B_MANIFEST.read_text(encoding="utf-8"))

dataset_lookup = {}
for row in phase2b_manifest["rows"]:
    key = (str(row["ratio"]), str(row["family"]), str(row["scenario"]))
    dataset_lookup[key] = row

missing = []
run_plan = []
for ratio, family, scenario, variant in TARGET_RUNS:
    key = (ratio, family, scenario)
    row = dataset_lookup.get(key)
    if row is None:
        missing.append(key)
        continue
    if str(row.get("status", "")) != "ready":
        missing.append(key)
        continue
    variant_cfg = VARIANT_LOOKUP[variant]
    run_plan.append({
        "ratio": ratio, "family": family, "scenario": scenario, "variant_id": variant,
        "dataset": str(LOCAL_PROJECT / row["output"]),
        "holdout_ref": str(LOCAL_PROJECT / "data" / "prepared" / "splits" / f"{family}_holdout.csv"),
        "sequence_length": int(variant_cfg["sequence_length"]),
        "g_pretrain_epochs": int(variant_cfg["generator_pretrain_epochs"]),
        "tokenizer_mode": variant_cfg["tokenizer_mode"],
        "generator_reward_mode": "on" if variant_cfg["sql_reward"] else "off",
    })

assert not missing, (
    f"Khong tim thay hoac chua 'ready' trong phase2b manifest cho: {missing}\n"
    "Kiem tra lai data/prepared/phase2b/dataset_manifest.json tren Drive."
)
assert len(run_plan) == 13

plan_df = pd.DataFrame(run_plan)
print(plan_df[["ratio", "family", "scenario", "variant_id", "sequence_length",
               "g_pretrain_epochs", "tokenizer_mode", "generator_reward_mode"]].to_string(index=False))


ratio  family scenario variant_id  sequence_length  g_pretrain_epochs tokenizer_mode generator_reward_mode
  100 boolean        E         V2              160                120      sql_aware                   off
  100    time        D         V8              160                160      sql_aware                    on
  100   union        D         V2              160                120      sql_aware                   off
  200   union        D         V2              160                120      sql_aware                   off
  500 boolean        D         V8              160                160      sql_aware                    on
  500 boolean        E         V2              160                120      sql_aware                   off
  500   error        B         V8              160                160      sql_aware                    on
  500   error        D         V4              160                160  raw_character                   off
  500   error        D         V8    

## 7. Vòng lặp chính: chạy tuần tự 13 run bằng `models.seqgan_improved.runtime.train` trực tiếp (không qua `run-matrix`, vì cần truyền `--batch-size 384` riêng cho từng run)

Mỗi run ghi vào `results_root/phase2b_batch6x/<family>/<scenario>/R<ratio>/<variant>_B6X/`, tách khỏi `results/phase2b/` chuẩn. Có bật `--checkpoint-dir` (lưu mỗi 5 epoch, giữ 3 bản gần nhất) — nếu Colab ngắt session giữa chừng, chạy lại đúng cell này (nó tự phát hiện checkpoint đã có và thêm `--resume-latest`).

In [ ]:
import concurrent.futures as cf
import threading

# So job chay DONG THOI. De ca 13 la mac dinh vi ban yeu cau chay song song --
# neu gap OOM / GPU qua tai, chi can giam so nay (vd 4) roi chay lai, cac run
# da co checkpoint se tu resume, khong mat tien do.
MAX_PARALLEL = 13

LOCAL_CHECKPOINT_ROOT = Path("/content/checkpoints_rerun_13")
DRIVE_CHECKPOINT_ROOT = RESULTS_ROOT / "phase2b_batch6x" / "_checkpoints_rerun_13"
LOCAL_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print("Checkpoint local (nhanh, tam):", LOCAL_CHECKPOINT_ROOT)
print("Checkpoint Drive (ben vung, backup that su):", DRIVE_CHECKPOINT_ROOT)
print(f"Chay song song toi da {MAX_PARALLEL} job cung luc")

_print_lock = threading.Lock()


def _safe_print(msg: str) -> None:
    with _print_lock:
        print(msg, flush=True)


def run_one(i: int, run: dict) -> dict:
    ratio, family, scenario, variant = run["ratio"], run["family"], run["scenario"], run["variant_id"]
    variant_tag = f"{variant}{VARIANT_SUFFIX}"
    run_id = f"phase2b__seqgan_improved__{family}__{scenario}__R{ratio}__{variant_tag}"
    run_key = f"{family}_{scenario}_R{ratio}_{variant_tag}"

    out_dir = RESULTS_ROOT / "phase2b_batch6x" / family / scenario / f"R{ratio}" / variant_tag
    local_ckpt_dir = LOCAL_CHECKPOINT_ROOT / run_key
    drive_ckpt_dir = DRIVE_CHECKPOINT_ROOT / run_key
    out_dir.mkdir(parents=True, exist_ok=True)
    local_ckpt_dir.mkdir(parents=True, exist_ok=True)
    drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

    has_local_ckpt = any(local_ckpt_dir.glob("adversarial_epoch_*.pt")) or (local_ckpt_dir / "latest_adversarial.pt").exists()
    has_drive_ckpt = any(drive_ckpt_dir.glob("adversarial_epoch_*.pt")) or (drive_ckpt_dir / "latest_adversarial.pt").exists()
    resume_flag = ["--resume-latest"] if (has_local_ckpt or has_drive_ckpt) else []

    _safe_print(
        f"[{i}/13] BAT DAU: {run_id} "
        f"batch-size={NEW_BATCH_SIZE}{'  [RESUME]' if resume_flag else ''}"
    )

    cmd = [
        sys.executable, "-m", "models.seqgan_improved.runtime.train",
        "--dataset", run["dataset"],
        "--holdout-ref", run["holdout_ref"],
        "--family", family,
        "--scenario", scenario,
        "--phase", "phase2b",
        "--ratio", ratio,
        "--variant-id", variant_tag,
        "--config", "seqgan_improved",
        "--tokenizer-mode", run["tokenizer_mode"],
        "--generator-reward-mode", run["generator_reward_mode"],
        "--reward-alpha", str(config["phase3"]["discriminator_reward_weight"]),
        "--sequence-length", str(run["sequence_length"]),
        "--g-pretrain-epochs", str(run["g_pretrain_epochs"]),
        "--d-pretrain-steps", str(BLOCK["discriminator_pretrain_steps"]),
        "--d-pretrain-epochs", str(BLOCK["discriminator_pretrain_epochs"]),
        "--adv-epochs", str(BLOCK["adversarial_epochs"]),
        "--g-steps", str(BLOCK["generator_steps"]),
        "--d-steps", str(BLOCK["discriminator_steps"]),
        "--d-epochs", str(BLOCK["discriminator_epochs"]),
        "--rollout-num", str(BLOCK["rollout_count"]),
        "--n-samples", str(config["generation"]["n_samples"]),
        "--seed", str(config["seed"]),
        "--batch-size", str(NEW_BATCH_SIZE),
        "--out-dir", str(out_dir),
        "--checkpoint-dir", str(local_ckpt_dir),
        "--checkpoint-copy-dir", str(drive_ckpt_dir),
        "--checkpoint-every", "10",
        "--checkpoint-keep", "3",
        *resume_flag,
    ]

    log_path = out_dir / "logs" / "train_rerun.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as handle:
        handle.write(f"\n\n===== NEW SESSION {datetime.now().isoformat()} =====\n")
        handle.flush()
        result = subprocess.run(cmd, cwd=LOCAL_PROJECT, stdout=handle, stderr=subprocess.STDOUT, text=True, check=False)

    status = "OK" if result.returncode == 0 else f"LOI (return_code={result.returncode})"
    drive_ckpt_files = sorted(drive_ckpt_dir.glob("*.pt"))

    if result.returncode != 0:
        _safe_print(f"[{i}/13] !!! LOI: {run_id} (code={result.returncode}) -- xem log: {log_path}")
    else:
        _safe_print(f"[{i}/13] >>> OK: {run_id} ({len(drive_ckpt_files)} checkpoint .pt tren Drive)")

    return {
        "run_id": run_id, "family": family, "scenario": scenario, "ratio": ratio,
        "variant_id": variant_tag, "returncode": result.returncode, "status": status,
        "out_dir": str(out_dir), "local_ckpt_dir": str(local_ckpt_dir),
        "drive_ckpt_dir": str(drive_ckpt_dir), "drive_checkpoint_count": len(drive_ckpt_files),
        "resumed": bool(resume_flag),
    }


ALL_RESULTS = [None] * len(run_plan)
with cf.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as executor:
    future_to_idx = {
        executor.submit(run_one, i, run): i
        for i, run in enumerate(run_plan, start=1)
    }
    for future in cf.as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            ALL_RESULTS[idx - 1] = future.result()
        except Exception as exc:
            _safe_print(f"[{idx}/13] !!! EXCEPTION khong bat duoc trong thread: {exc}")
            ALL_RESULTS[idx - 1] = {
                "run_id": f"run_{idx}", "returncode": -1, "status": f"EXCEPTION: {exc}",
                "resumed": False, "drive_checkpoint_count": 0,
            }

print("#" * 70)
ok_count = sum(1 for r in ALL_RESULTS if r and r["returncode"] == 0)
print(f"HOAN TAT {ok_count}/{len(ALL_RESULTS)} run (chay song song, toi da {MAX_PARALLEL} job cung luc)")
print("#" * 70)


Checkpoint local (nhanh, tam): /content/checkpoints_rerun_13
Checkpoint Drive (ben vung, backup that su): /content/drive/MyDrive/GAN/GAN_for_SQLi/result/phase2b_batch6x/_checkpoints_rerun_13
Chay song song toi da 13 job cung luc
[1/13] BAT DAU: phase2b__seqgan_improved__boolean__E__R100__V2_B6X batch-size=384
[2/13] BAT DAU: phase2b__seqgan_improved__time__D__R100__V8_B6X batch-size=384
[4/13] BAT DAU: phase2b__seqgan_improved__union__D__R200__V2_B6X batch-size=384
[3/13] BAT DAU: phase2b__seqgan_improved__union__D__R100__V2_B6X batch-size=384
[5/13] BAT DAU: phase2b__seqgan_improved__boolean__D__R500__V8_B6X batch-size=384
[9/13] BAT DAU: phase2b__seqgan_improved__error__D__R500__V8_B6X batch-size=384
[6/13] BAT DAU: phase2b__seqgan_improved__boolean__E__R500__V2_B6X batch-size=384
[7/13] BAT DAU: phase2b__seqgan_improved__error__B__R500__V8_B6X batch-size=384
[10/13] BAT DAU: phase2b__seqgan_improved__time__A__R500__V8_B6X batch-size=384
[8/13] BAT DAU: phase2b__seqgan_improved__erro

## 7b. Xác minh checkpoint đã thực sự ghi lên Drive cho từng run (chạy sau khi vòng lặp trên xong hoặc bị ngắt giữa chừng)


In [ ]:
# Kiem tra checkpoint BEN VUNG tren Drive (checkpoint-copy-dir) cho tung run --
# day moi la bang chung "backup" thuc su song sot qua mat runtime, khong phai
# ban local o /content.
report_rows = []
for run in run_plan:
    ratio, family, scenario, variant = run["ratio"], run["family"], run["scenario"], run["variant_id"]
    variant_tag = f"{variant}{VARIANT_SUFFIX}"
    run_key = f"{family}_{scenario}_R{ratio}_{variant_tag}"
    drive_ckpt_dir = DRIVE_CHECKPOINT_ROOT / run_key
    pt_files = sorted(drive_ckpt_dir.glob("*.pt")) if drive_ckpt_dir.exists() else []
    latest = drive_ckpt_dir / "latest_adversarial.pt"
    epoch_ckpts = sorted(drive_ckpt_dir.glob("adversarial_epoch_*.pt"))
    last_epoch = None
    if epoch_ckpts:
        try:
            last_epoch = max(int(p.stem.split("_")[-1]) for p in epoch_ckpts)
        except ValueError:
            last_epoch = None
    report_rows.append({
        "run": f"{family}/{scenario}/R{ratio}/{variant_tag}",
        "drive_ckpt_dir": str(drive_ckpt_dir),
        "num_pt_files": len(pt_files),
        "has_latest_adversarial": latest.exists(),
        "last_epoch_checkpoint": last_epoch,
    })

df_ckpt = pd.DataFrame(report_rows)
missing_backup = df_ckpt[df_ckpt["num_pt_files"] == 0]
if len(missing_backup) > 0:
    print(f"CANH BAO: {len(missing_backup)} run CHUA co bat ky checkpoint .pt nao tren Drive:")
    print(missing_backup["run"].tolist())
else:
    print("Tat ca run co it nhat 1 checkpoint .pt tren Drive (backup ben vung).")

df_ckpt


## 8. Tổng hợp kết quả 13 run — đọc `training_metadata.json` từng run, lưu bảng tóm tắt vào `results/phase2b_batch6x/rerun_13_summary.csv`

In [ ]:
summary_rows = []
for entry in ALL_RESULTS:
    metadata_path = Path(entry["out_dir"]) / "training_metadata.json"
    stop_reason = ""
    adv_epochs_completed = ""
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        stop_reason = metadata.get("stop_reason", "")
        adv_training = metadata.get("adversarial_training", {})
        if isinstance(adv_training, dict):
            adv_epochs_completed = adv_training.get("epochs_completed", adv_training.get("epoch", ""))
    summary_rows.append({**entry, "stop_reason": stop_reason, "adv_epochs_completed": adv_epochs_completed})

summary_df = pd.DataFrame(summary_rows)
print(summary_df[["run_id", "family", "scenario", "ratio", "variant_id",
                   "status", "adv_epochs_completed", "stop_reason", "resumed"]].to_string(index=False))

SUMMARY_PATH = RESULTS_ROOT / "phase2b_batch6x" / "rerun_13_summary.csv"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
print()
print("Da ghi bang tong hop:", SUMMARY_PATH)
print()
print("So run OK:", (summary_df["returncode"] == 0).sum(), "/", len(summary_df))
